# 01 — Data Processing & Cleaning with PySpark

**Project:** Movie Popularity Prediction Using Big Data and Machine Learning

This is the **single, complete PySpark data-processing notebook** of the project. Every step from
loading the raw CSV files to saving the final processed dataset happens here:

| # | Section | Main Spark features used |
|---|---------|--------------------------|
| 1 | Spark initialization | `SparkSession` |
| 2 | Data collection (download + attribution) | — |
| 3 | Dataset loading | `spark.read.csv` with explicit options |
| 4 | Data inspection | `printSchema()`, `show()`, `count()`, `describe()`, null profiling with `agg()` |
| 5 | Data cleaning | `cast`, `filter`, `dropDuplicates`, `when/otherwise`, `approxQuantile` imputation |
| 6 | Transformation & feature engineering | `regexp_extract_all`, `size`, `array_contains`, date functions, `log1p` |
| 7 | Spark analytics operations | `select`, `filter`, `groupBy().agg()`, `explode`, **Spark SQL**, **joins**, window functions |
| 8 | Validation | row counts, schema, nulls, duplicates, value ranges with `assert` |
| 9 | Save processed data | Parquet in `data/processed/` |

**Output (input to notebooks 02 and 03 and the dashboard):** `data/processed/movies_processed.parquet`

**Why Spark?** The raw `movies_metadata.csv` contains multi-line quoted text, mixed types, malformed rows and
JSON-like nested columns. Spark's DataFrame API lets us express the whole cleaning pipeline declaratively and
the exact same code would scale unchanged to the full MovieLens ratings file (26M rows / 700 MB) or a cluster.

## 1. Spark Initialization

**Installation:** `pip install -r requirements.txt` installs `pyspark`. Spark also needs **Java 17 or 21**.
Set `JAVA_HOME` to your JDK. For convenience, the cell below also auto-detects a portable JDK unpacked into the
project's `.tools/` folder (see README).

In [1]:
import os, sys, glob, shutil, urllib.request
from pathlib import Path

# ---- Project paths (the notebook lives in notebooks/) ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ---- Java detection: use JAVA_HOME if set, otherwise look for a portable JDK in .tools/ ----
if not os.environ.get("JAVA_HOME"):
    candidates = sorted(glob.glob(str(PROJECT_ROOT / ".tools" / "jdk*")))
    if candidates:
        os.environ["JAVA_HOME"] = candidates[-1]
if os.environ.get("JAVA_HOME"):
    os.environ["PATH"] = str(Path(os.environ["JAVA_HOME"]) / "bin") + os.pathsep + os.environ["PATH"]
assert shutil.which("java"), "Java not found. Install JDK 17/21 and set JAVA_HOME (see README)."

# Make Spark workers use the same Python interpreter as this notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Project root:", PROJECT_ROOT)
print("JAVA_HOME   :", os.environ.get("JAVA_HOME"))

Project root: d:\Big Data\Movie-Popularity-Prediction
JAVA_HOME   : None


In [2]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import IntegerType, DoubleType, LongType

spark = (
    SparkSession.builder
    .appName("MoviePopularityProcessing")
    .master("local[*]")                              # use all local cores
    .config("spark.sql.shuffle.partitions", "8")     # small data -> few shuffle partitions
    .config("spark.driver.memory", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.3.0


## 2. Data Collection

**Dataset:** *The Movies Dataset* by **Rounak Banik** — Kaggle:
<https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset> (metadata collected from the TMDB API and
GroupLens MovieLens; licence CC0: Public Domain).

Kaggle requires a login to download, so this project uses a public byte-identical mirror of the same files on
Hugging Face (<https://huggingface.co/datasets/gaygaaa/TheMoviesDataset>). You may also download the ZIP from
Kaggle manually and place the CSV files into `data/raw/`.

| File | Size | Used for |
|------|------|----------|
| `movies_metadata.csv` | 34 MB, ~45k movies, 24 columns | **Main dataset** (features + target) |
| `links.csv` | 0.9 MB | Join key: MovieLens `movieId` ↔ TMDB `tmdbId` |
| `ratings_small.csv` | 2.4 MB, 100k ratings | Optional post-release analysis only (**never** a model input) |

In [3]:
MIRROR = "https://huggingface.co/datasets/gaygaaa/TheMoviesDataset/resolve/main/"
RAW_FILES = ["movies_metadata.csv", "links.csv", "ratings_small.csv"]

for name in RAW_FILES:
    target = RAW_DIR / name
    if target.exists():
        print(f"[ok]       {name:22s} {target.stat().st_size/1e6:7.1f} MB (already present)")
    else:
        print(f"[download] {name} ...")
        urllib.request.urlretrieve(MIRROR + name, target)
        print(f"           saved {target.stat().st_size/1e6:.1f} MB")

[download] movies_metadata.csv ...
           saved 34.4 MB
[download] links.csv ...
           saved 0.9 MB
[download] ratings_small.csv ...
           saved 2.3 MB


## 3. Dataset Loading

`movies_metadata.csv` has overview texts with embedded commas, quotes and line breaks, so we read it with
`multiLine=True` and `escape='"'`. All columns are read as **strings** first (`inferSchema=False`) so that we
control type conversion explicitly in the cleaning step and can count the values that fail to convert.

In [4]:
movies_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("quote", '"')
    .option("inferSchema", False)
    .csv(str(RAW_DIR / "movies_metadata.csv"))
)
links_raw = spark.read.csv(str(RAW_DIR / "links.csv"), header=True, inferSchema=True)
ratings_raw = spark.read.csv(str(RAW_DIR / "ratings_small.csv"), header=True, inferSchema=True)

print("movies_metadata rows:", movies_raw.count(), "| columns:", len(movies_raw.columns))
print("links rows          :", links_raw.count())
print("ratings_small rows  :", ratings_raw.count())

movies_metadata rows: 45466 | columns: 24
links rows          : 45843
ratings_small rows  : 100004


## 4. Data Inspection

In [5]:
movies_raw.printSchema()

root
 |-- adult: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- homepage: string (nullable = true)
 |-- id: string (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- video: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nu

In [6]:
movies_raw.select("id", "title", "budget", "genres", "release_date", "runtime",
                  "original_language", "popularity", "vote_count").show(5, truncate=40)

+-----+---------------------------+--------+----------------------------------------+------------+-------+-----------------+----------+----------+
|   id|                      title|  budget|                                  genres|release_date|runtime|original_language|popularity|vote_count|
+-----+---------------------------+--------+----------------------------------------+------------+-------+-----------------+----------+----------+
|  862|                  Toy Story|30000000|[{'id': 16, 'name': 'Animation'}, {'i...|  1995-10-30|   81.0|               en| 21.946943|      5415|
| 8844|                    Jumanji|65000000|[{'id': 12, 'name': 'Adventure'}, {'i...|  1995-12-15|  104.0|               en| 17.015539|      2413|
|15602|           Grumpier Old Men|       0|[{'id': 10749, 'name': 'Romance'}, {'...|  1995-12-22|  101.0|               en|   11.7129|        92|
|31357|          Waiting to Exhale|16000000|[{'id': 35, 'name': 'Comedy'}, {'id':...|  1995-12-22|  127.0|            

In [7]:
# Summary statistics of the (still string-typed) numeric columns. Note the min/max are compared as TEXT and
# contain garbage such as file paths and titles -> proof that explicit type conversion is required.
movies_raw.select("budget", "popularity", "runtime", "revenue", "vote_count").describe().show()

+-------+--------------------+--------------------+------------------+--------------------+------------------+
|summary|              budget|          popularity|           runtime|             revenue|        vote_count|
+-------+--------------------+--------------------+------------------+--------------------+------------------+
|  count|               45466|               45461|             45203|               45460|             45460|
|   mean|   4224578.813474693|  2.9214783449186243| 94.12819945578833|1.1209348539265288E7|109.89733831940167|
| stddev|1.7424132634341173E7|   6.005414309041766|38.407810485508506| 6.433224674065125E7|491.31037393965795|
|    min|/ff9qCepilowshEtG...|                 0.0|               0.0|                   0|                 0|
|    max|              998000|Beware Of Frost B...|              99.0|            99965753|               998|
+-------+--------------------+--------------------+------------------+--------------------+------------------+



### 4.1 Missing values per column (Spark aggregation)

In [8]:
total = movies_raw.count()
null_counts = movies_raw.agg(*[
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in movies_raw.columns
])
null_pdf = null_counts.toPandas().T.rename(columns={0: "missing"})
null_pdf["missing_%"] = (null_pdf["missing"] / total * 100).round(2)
null_pdf.sort_values("missing", ascending=False)

d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,missing,missing_%
belongs_to_collection,40972,90.12
homepage,37684,82.88
tagline,25055,55.11
overview,959,2.11
poster_path,386,0.85
runtime,263,0.58
status,87,0.19
release_date,87,0.19
imdb_id,17,0.04
original_language,11,0.02


### 4.2 Data-quality problems detected

In [9]:
# (a) malformed rows: the 'id' column must be numeric. A few rows in this file are shifted (id holds a date).
bad_id = movies_raw.filter(~F.col("id").rlike(r"^\d+$"))
print("Rows with non-numeric id:", bad_id.count())
bad_id.select("id", "title", "budget", "popularity").show(truncate=30)

# (b) duplicate movie ids
dup_ids = movies_raw.groupBy("id").count().filter("count > 1")
print("Duplicated ids:", dup_ids.count(), "| extra rows:", dup_ids.agg(F.sum(F.col("count") - 1)).first()[0])

# (c) zeros used as 'unknown' for budget / revenue / runtime
for c in ["budget", "revenue", "runtime"]:
    print(f"{c:8s} == 0 :", movies_raw.filter(F.expr(f"try_cast({c} AS DOUBLE)") == 0).count())

# (d) release status distribution
movies_raw.groupBy("status").count().orderBy(F.desc("count")).show()

Rows with non-numeric id: 3
+----------+-----+------------------------------+---------------------+
|        id|title|                        budget|           popularity|
+----------+-----+------------------------------+---------------------+
|1997-08-20| NULL|/ff9qCepilowshEtG2GYWwzt2bs...|                 NULL|
|2012-09-29| NULL|/zV8bHuSL6WXoD6FWogP9j4x80b...|                 NULL|
|2014-01-01| NULL|/zaSf5OG7V8X8gqFvly88zDdRm4...|Beware Of Frost Bites|
+----------+-----+------------------------------+---------------------+

Duplicated ids: 29 | extra rows: 30
budget   == 0 : 36573
revenue  == 0 : 38052
runtime  == 0 : 1558
+---------------+-----+
|         status|count|
+---------------+-----+
|       Released|45014|
|        Rumored|  230|
|Post Production|   98|
|           NULL|   87|
|  In Production|   20|
|        Planned|   15|
|       Canceled|    2|
+---------------+-----+



## 5. Data Cleaning

Cleaning rules (each step prints how many rows it affects):

1. **Type conversion** — cast numeric columns; `try_cast` semantics make unparseable values `null`.
2. **Malformed rows** — drop rows whose `id` is not a valid integer.
3. **Duplicates** — keep one row per TMDB `id`.
4. **Scope filter** — keep released, non-adult movies with a valid release date and title.
5. **Invalid target** — drop rows with missing or non-positive `popularity`.
6. **Invalid budget** — budgets below \$1,000 are placeholders (`0`, `1`, `5`…) → treated as *unknown* (`null`).
7. **Invalid runtime** — `0`, negative or > 400 min → `null`, then imputed with the median.
8. **Revenue** — `0` → `null` (EDA only, never a model feature).

In [10]:
step_counts = {"raw": movies_raw.count()}

# 1) Type conversion. Only rows that already have a numeric id are cast, malformed rows are removed in step 2.
movies = (
    movies_raw
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .withColumn("budget", F.expr("try_cast(budget AS DOUBLE)"))
    .withColumn("revenue", F.expr("try_cast(revenue AS DOUBLE)"))
    .withColumn("popularity", F.expr("try_cast(popularity AS DOUBLE)"))
    .withColumn("runtime", F.expr("try_cast(runtime AS DOUBLE)"))
    .withColumn("vote_average", F.expr("try_cast(vote_average AS DOUBLE)"))
    .withColumn("vote_count", F.expr("try_cast(vote_count AS DOUBLE)"))
    .withColumn("release_date", F.to_date(F.expr("try_to_timestamp(release_date, 'yyyy-MM-dd')")))
    .withColumn("title", F.trim("title"))
)

# 2) Malformed rows
movies = movies.filter(F.col("id").isNotNull())
step_counts["valid_id"] = movies.count()

# 3) Duplicates (same TMDB id appears more than once)
movies = movies.dropDuplicates(["id"])
step_counts["deduplicated"] = movies.count()

# 4) Scope: released, non-adult, with date and title
movies = movies.filter(
    (F.col("status") == "Released")
    & (F.col("adult") == "False")
    & F.col("release_date").isNotNull()
    & F.col("title").isNotNull() & (F.col("title") != "")
)
step_counts["released_with_date"] = movies.count()

# 5) Valid target
movies = movies.filter(F.col("popularity").isNotNull() & (F.col("popularity") > 0))
step_counts["valid_popularity"] = movies.count()

for k, v in step_counts.items():
    print(f"{k:22s} {v:7d}")

raw                      45466
valid_id                 45463
deduplicated             45433
released_with_date       44899
valid_popularity         44861


In [11]:
# 6) Budget placeholders -> unknown ; 8) revenue 0 -> unknown
movies = (
    movies
    .withColumn("budget", F.when(F.col("budget") >= 1000, F.col("budget")))
    .withColumn("revenue", F.when(F.col("revenue") > 0, F.col("revenue")))
)

# 7) Runtime: invalid -> null, then median imputation (approxQuantile is a distributed Spark algorithm)
movies = movies.withColumn(
    "runtime", F.when((F.col("runtime") > 0) & (F.col("runtime") <= 400), F.col("runtime"))
)
runtime_missing = movies.filter(F.col("runtime").isNull()).count()
runtime_median = movies.approxQuantile("runtime", [0.5], 0.001)[0]
movies = (
    movies
    .withColumn("runtime_imputed", F.col("runtime").isNull().cast("int"))
    .fillna({"runtime": runtime_median})
)
# original_language: tiny number of nulls -> 'xx' (TMDB code for 'no language')
movies = movies.fillna({"original_language": "xx"})

print("Runtime values imputed with median", runtime_median, "min:", runtime_missing)
print("Movies with known budget:", movies.filter(F.col("budget").isNotNull()).count())

Runtime values imputed with median 95.0 min: 1756
Movies with known budget: 8609


## 6. Data Transformation & Feature Engineering

The nested columns (`genres`, `production_companies`, `production_countries`, `belongs_to_collection`) are
stored as Python-style JSON strings such as `"[{'id': 16, 'name': 'Animation'}, ...]"`.
We parse them **inside Spark** with `regexp_extract_all` instead of Python UDFs (faster, no serialization).

### Prediction scenario and data-leakage policy
We predict popularity **before release**, so only information known before release is turned into model
features: budget, genres, runtime, release year/month, original language, franchise membership and production
companies/countries. `revenue`, `vote_count`, `vote_average` and user ratings are kept **for EDA only**.

In [12]:
GENRE_NAME_RE = r"'name': '([^']*)'"          # genre names
COMPANY_ID_RE = r"'id': (\d+)"                   # one match per production company
COUNTRY_RE = r"'iso_3166_1': '([A-Z]+)'"         # ISO country codes
movies = (
    movies
    # --- date features
    .withColumn("release_year", F.year("release_date"))
    .withColumn("release_month", F.month("release_date"))
    .withColumn("release_decade", (F.floor(F.col("release_year") / 10) * 10).cast("int"))
    # --- genres: array of names
    .withColumn("genres_list", F.regexp_extract_all("genres", F.lit(GENRE_NAME_RE), 1))
    .withColumn("num_genres", F.size("genres_list"))
    .withColumn("primary_genre", F.when(F.col("num_genres") > 0, F.col("genres_list")[0]).otherwise("Unknown"))
    .withColumn("genres_str", F.concat_ws("|", "genres_list"))
    # --- production companies / countries
    .withColumn("num_companies", F.size(F.regexp_extract_all("production_companies", F.lit(COMPANY_ID_RE), 1)))
    .withColumn("countries_list", F.regexp_extract_all("production_countries", F.lit(COUNTRY_RE), 1))
    .withColumn("num_countries", F.size("countries_list"))
    .withColumn("is_us_production", F.array_contains("countries_list", "US").cast("int"))
    # --- franchise / collection membership (known before release)
    .withColumn("has_collection", F.col("belongs_to_collection").isNotNull().cast("int"))
    # --- budget features: log scale + 'known' flag (unknown budget is information, not an error)
    .withColumn("budget_known", F.col("budget").isNotNull().cast("int"))
    .withColumn("log_budget", F.coalesce(F.log1p("budget"), F.lit(0.0)))
    # --- target on log scale (popularity is extremely right-skewed)
    .withColumn("log_popularity", F.log1p("popularity"))
)
movies.select("title", "genres_list", "primary_genre", "num_companies", "countries_list",
              "has_collection", "log_budget", "release_year", "release_month").show(5, truncate=35)

+------------------+-------------------------+-------------+-------------+-----------------------------------+--------------+------------------+------------+-------------+
|             title|              genres_list|primary_genre|num_companies|                     countries_list|has_collection|        log_budget|release_year|release_month|
+------------------+-------------------------+-------------+-------------+-----------------------------------+--------------+------------------+------------+-------------+
|    Judgment Night|[Action, Thriller, Crime]|       Action|            3|                           [JP, US]|             0|               0.0|        1993|           10|
|      Finding Nemo|      [Animation, Family]|    Animation|            1|                               [US]|             1|18.358805350872576|        2003|            5|
|      Forrest Gump| [Comedy, Drama, Romance]|       Comedy|            1|                               [US]|             0|17.822843761378

**One-hot (multi-hot) genre flags.** A movie can have several genres, so we create one 0/1 column per genre.
We only keep the official TMDB genres that appear at least 100 times (this removes a handful of junk values
that leaked from malformed source rows).

In [13]:
genre_counts = (
    movies.select(F.explode("genres_list").alias("genre"))
    .groupBy("genre").count()
    .filter(F.col("count") >= 100)
    .orderBy(F.desc("count"))
)
GENRES = [r["genre"] for r in genre_counts.collect()]
print(len(GENRES), "genres:", GENRES)

def genre_col(g):
    # 'Science Fiction' -> 'genre_science_fiction'
    return "genre_" + g.lower().replace(" ", "_")

for g in GENRES:
    movies = movies.withColumn(genre_col(g), F.array_contains("genres_list", g).cast("int"))

GENRE_COLS = [genre_col(g) for g in GENRES]
movies = movies.cache()
print("Rows:", movies.count())

20 genres: ['Drama', 'Comedy', 'Thriller', 'Romance', 'Action', 'Horror', 'Crime', 'Documentary', 'Adventure', 'Science Fiction', 'Family', 'Mystery', 'Fantasy', 'Animation', 'Foreign', 'Music', 'History', 'War', 'Western', 'TV Movie']
Rows: 44861


## 7. Spark Analytics Operations

### 7.1 `select` + `filter` — big-budget franchise films

In [14]:
(movies
 .filter((F.col("budget") >= 150_000_000) & (F.col("has_collection") == 1))
 .select("title", "release_year", (F.col("budget") / 1e6).alias("budget_musd"), F.round("popularity", 1).alias("popularity"))
 .orderBy(F.desc("popularity"))
 .show(10, truncate=40))

+----------------------------------------+------------+-----------+----------+
|                                   title|release_year|budget_musd|popularity|
+----------------------------------------+------------+-----------+----------+
|          Guardians of the Galaxy Vol. 2|        2017|      200.0|     185.3|
|                                  Avatar|        2009|      237.0|     185.1|
|          War for the Planet of the Apes|        2017|      152.0|     146.2|
|              Captain America: Civil War|        2016|      250.0|     145.9|
|Pirates of the Caribbean: Dead Men Te...|        2017|      230.0|     133.8|
|                         The Dark Knight|        2008|      185.0|     123.2|
|                            The Avengers|        2012|      220.0|      89.9|
|          Dawn of the Planet of the Apes|        2014|      170.0|      75.4|
|                 Guardians of the Galaxy|        2014|      170.0|      53.3|
|                 The Fate of the Furious|        20

### 7.2 `explode` + `groupBy` + `agg` — popularity by genre

In [15]:
genre_stats = (
    movies
    .select(F.explode("genres_list").alias("genre"), "popularity", "budget", "runtime")
    .filter(F.col("genre").isin(GENRES))
    .groupBy("genre")
    .agg(
        F.count("*").alias("movies"),
        F.round(F.avg("popularity"), 3).alias("avg_popularity"),
        F.round(F.percentile_approx("popularity", 0.5), 3).alias("median_popularity"),
        F.round(F.avg("budget") / 1e6, 2).alias("avg_budget_musd"),
        F.round(F.avg("runtime"), 1).alias("avg_runtime"),
    )
    .orderBy(F.desc("median_popularity"))
)
genre_stats.show(25)

+---------------+------+--------------+-----------------+---------------+-----------+
|          genre|movies|avg_popularity|median_popularity|avg_budget_musd|avg_runtime|
+---------------+------+--------------+-----------------+---------------+-----------+
|      Adventure|  3467|         6.005|             2.98|          52.11|      102.4|
|        Fantasy|  2290|         5.361|            2.535|          47.91|       95.1|
|       Thriller|  7562|         4.508|            2.358|          23.72|      100.4|
|         Family|  2738|         4.769|            2.312|          47.41|       89.3|
|         Action|  6539|         4.786|            2.224|          37.39|      102.4|
|      Animation|  1911|         4.748|            2.212|          48.09|       65.5|
|Science Fiction|  3010|         5.004|            2.188|          35.81|       95.2|
|          Crime|  4277|         4.157|            2.017|           22.4|      101.4|
|         Horror|  4628|         3.585|             1.

### 7.3 Yearly release trend (`groupBy` on a derived column)

In [16]:
yearly_stats = (
    movies.groupBy("release_year")
    .agg(F.count("*").alias("movies"),
         F.round(F.avg("popularity"), 3).alias("avg_popularity"),
         F.round(F.percentile_approx("popularity", 0.5), 3).alias("median_popularity"),
         F.round(F.avg("budget") / 1e6, 2).alias("avg_budget_musd"))
    .orderBy("release_year")
)
yearly_stats.filter("release_year >= 2005").show(15)

+------------+------+--------------+-----------------+---------------+
|release_year|movies|avg_popularity|median_popularity|avg_budget_musd|
+------------+------+--------------+-----------------+---------------+
|        2005|  1114|         3.149|            1.341|          26.67|
|        2006|  1259|         3.242|            1.396|          25.08|
|        2007|  1313|         3.174|              1.3|          23.78|
|        2008|  1456|         3.228|            1.439|          24.22|
|        2009|  1573|         3.234|            1.392|          24.71|
|        2010|  1477|         2.994|            1.271|          24.57|
|        2011|  1644|         3.021|            1.335|          22.45|
|        2012|  1707|         3.076|            1.426|          23.48|
|        2013|  1879|         3.308|            1.597|          24.26|
|        2014|  1960|         3.897|            1.582|          23.92|
|        2015|  1873|         3.576|            1.464|          24.34|
|     

### 7.4 Spark SQL

In [17]:
movies.createOrReplaceTempView("movies")

# Popularity by budget tier (CASE WHEN) — compares known-budget tiers with unknown budget
spark.sql('''
    SELECT CASE
             WHEN budget IS NULL          THEN '0) unknown'
             WHEN budget <  1000000       THEN '1) < $1M'
             WHEN budget < 10000000       THEN '2) $1M - 10M'
             WHEN budget < 50000000       THEN '3) $10M - 50M'
             WHEN budget < 100000000      THEN '4) $50M - 100M'
             ELSE                              '5) >= $100M'
           END                                   AS budget_tier,
           COUNT(*)                              AS movies,
           ROUND(AVG(popularity), 2)             AS avg_popularity,
           ROUND(percentile_approx(popularity, 0.5), 2) AS median_popularity
    FROM movies
    GROUP BY 1
    ORDER BY 1
''').show()

+--------------+------+--------------+-----------------+
|   budget_tier|movies|avg_popularity|median_popularity|
+--------------+------+--------------+-----------------+
|    0) unknown| 36252|          1.86|             0.87|
|      1) < $1M|  1229|          2.84|             1.32|
|  2) $1M - 10M|  3141|          5.23|             4.07|
| 3) $10M - 50M|  3067|          8.78|             8.03|
|4) $50M - 100M|   791|         12.56|            10.27|
|   5) >= $100M|   381|         21.38|            14.89|
+--------------+------+--------------+-----------------+



In [18]:
# Top original languages by number of films and their popularity
spark.sql('''
    SELECT original_language, COUNT(*) AS movies,
           ROUND(AVG(popularity), 2) AS avg_popularity,
           ROUND(percentile_approx(popularity, 0.5), 2) AS median_popularity
    FROM movies
    GROUP BY original_language
    HAVING COUNT(*) >= 300
    ORDER BY movies DESC
''').show()

# Popularity by release month
spark.sql('''
    SELECT release_month, COUNT(*) AS movies, ROUND(percentile_approx(popularity, 0.5), 3) AS median_popularity
    FROM movies GROUP BY release_month ORDER BY release_month
''').show(12)

+-----------------+------+--------------+-----------------+
|original_language|movies|avg_popularity|median_popularity|
+-----------------+------+--------------+-----------------+
|               en| 31844|           3.4|             1.28|
|               fr|  2414|          2.47|             1.36|
|               it|  1506|           1.7|              0.8|
|               ja|  1338|          2.24|              1.2|
|               de|  1071|          1.66|             0.85|
|               es|   981|          1.81|             0.84|
|               ru|   815|          1.18|             0.72|
|               hi|   500|          1.74|             1.11|
|               ko|   442|          2.31|             1.59|
|               zh|   405|          2.33|             1.27|
|               sv|   382|          1.66|             0.88|
|               pt|   313|          1.27|             0.64|
|               cn|   309|           3.1|             1.76|
+-----------------+------+--------------

### 7.5 Window function — most popular movie of each recent year

In [19]:
w = Window.partitionBy("release_year").orderBy(F.desc("popularity"))
(movies
 .withColumn("rank_in_year", F.rank().over(w))
 .filter((F.col("rank_in_year") == 1) & (F.col("release_year") >= 2005))
 .select("release_year", "title", F.round("popularity", 1).alias("popularity"))
 .orderBy("release_year")
 .show(15, truncate=45))

+------------+------------------------------------------+----------+
|release_year|                                     title|popularity|
+------------+------------------------------------------+----------+
|        2005|                             Batman Begins|      28.5|
|        2006|Pirates of the Caribbean: Dead Man's Chest|      29.5|
|        2007|                       The Mother of Tears|      34.1|
|        2008|                           The Dark Knight|     123.2|
|        2009|                                    Avatar|     185.1|
|        2010|                The Twilight Saga: Eclipse|      34.0|
|        2011|            Rise of the Planet of the Apes|      41.6|
|        2012|                              The Avengers|      89.9|
|        2013|                      Thor: The Dark World|      34.9|
|        2014|                                Big Hero 6|     213.8|
|        2015|                                   Minions|     547.5|
|        2016|                    

### 7.6 Joins — MovieLens ratings → links → movies (optional post-release analysis)

`ratings_small.csv` uses MovieLens `movieId`; `links.csv` maps it to the TMDB `id`. We aggregate the ratings per
movie and **join** them to the cleaned movies. These columns are **post-release** information and are saved to a
separate file used only in EDA — they are never model inputs.

In [20]:
ratings_per_movie = (
    ratings_raw.groupBy("movieId")
    .agg(F.count("*").alias("ml_rating_count"), F.round(F.avg("rating"), 3).alias("ml_avg_rating"))
)
links = links_raw.select("movieId", F.col("tmdbId").cast(IntegerType()).alias("tmdbId")).dropna()

movie_ratings = (
    ratings_per_movie
    .join(links, on="movieId", how="inner")                                   # movieId -> tmdbId
    .join(movies.select("id", "title", "popularity", "vote_count", "vote_average", "primary_genre"),
          F.col("tmdbId") == F.col("id"), how="inner")                          # tmdbId -> movie
    .drop("tmdbId")
)
print("Movies with MovieLens ratings after join:", movie_ratings.count())
movie_ratings.orderBy(F.desc("ml_rating_count")).show(8, truncate=35)
print("corr(ml_rating_count, popularity) =", round(movie_ratings.stat.corr("ml_rating_count", "popularity"), 3))

Movies with MovieLens ratings after join: 9004
+-------+---------------+-------------+---+------------------------+----------+----------+------------+-------------+
|movieId|ml_rating_count|ml_avg_rating| id|                   title|popularity|vote_count|vote_average|primary_genre|
+-------+---------------+-------------+---+------------------------+----------+----------+------------+-------------+
|    356|            341|        4.054| 13|            Forrest Gump| 48.307194|    8147.0|         8.2|       Comedy|
|    296|            324|        4.256|680|            Pulp Fiction|140.950236|    8670.0|         8.3|     Thriller|
|    318|            311|        4.487|278|The Shawshank Redemption| 51.645403|    8358.0|         8.5|        Drama|
|    593|            304|        4.138|274|The Silence of the Lambs|  4.307222|    4549.0|         8.1|        Crime|
|    260|            291|        4.222| 11|               Star Wars| 42.149697|    6778.0|         8.1|    Adventure|
|    480|

## 8. Validation

In [21]:
FINAL_COLUMNS = (
    ["id", "imdb_id", "title", "overview", "release_date", "release_year", "release_month", "release_decade",
     "original_language", "genres_str", "primary_genre", "num_genres",
     "budget", "budget_known", "log_budget", "runtime", "runtime_imputed",
     "has_collection", "num_companies", "num_countries", "is_us_production"]
    + GENRE_COLS
    + ["revenue", "vote_count", "vote_average",          # post-release: EDA only
       "popularity", "log_popularity"]                    # target
)
final_df = movies.select(*FINAL_COLUMNS)
final_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_month: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- original_language: string (nullable = false)
 |-- genres_str: string (nullable = false)
 |-- primary_genre: string (nullable = true)
 |-- num_genres: integer (nullable = true)
 |-- budget: double (nullable = true)
 |-- budget_known: integer (nullable = false)
 |-- log_budget: double (nullable = false)
 |-- runtime: double (nullable = false)
 |-- runtime_imputed: integer (nullable = false)
 |-- has_collection: integer (nullable = false)
 |-- num_companies: integer (nullable = true)
 |-- num_countries: integer (nullable = true)
 |-- is_us_production: integer (nullable = true)
 |-- genre_drama: integer (nullable = true)
 |-- genre_comedy: integer (nullab

In [22]:
n_rows = final_df.count()
n_ids = final_df.select("id").distinct().count()

# Missing values in columns that must be complete (budget/revenue/overview/imdb_id may legitimately be null)
must_be_complete = [c for c in FINAL_COLUMNS if c not in ("budget", "revenue", "overview", "imdb_id",
                                                           "vote_count", "vote_average")]
nulls = final_df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in must_be_complete]).first().asDict()
bad = {k: v for k, v in nulls.items() if v}

ranges = final_df.agg(
    F.min("release_year").alias("min_year"), F.max("release_year").alias("max_year"),
    F.min("runtime").alias("min_runtime"), F.max("runtime").alias("max_runtime"),
    F.min("budget").alias("min_budget"), F.min("popularity").alias("min_pop"), F.max("popularity").alias("max_pop"),
).first()

print("rows:", n_rows, "| distinct ids:", n_ids)
print("nulls in required columns:", bad or "none")
print(ranges)

assert n_rows == n_ids, "duplicate ids remain"
assert not bad, f"unexpected nulls: {bad}"
assert 1 <= ranges.min_runtime and ranges.max_runtime <= 400
assert ranges.min_budget is None or ranges.min_budget >= 1000
assert ranges.min_pop > 0
print("All validation checks passed.")

rows: 44861 | distinct ids: 44861
nulls in required columns: none
Row(min_year=1874, max_year=2017, min_runtime=1.0, max_runtime=400.0, min_budget=1000.0, min_pop=1e-06, max_pop=547.488298)
All validation checks passed.


## 9. Save Processed Data

The processed data is small (tens of thousands of rows), so we collect it with `toPandas()` and write
**single-file Parquet** with `pyarrow`. A single file is easier to load from the dashboard and to deploy on
Streamlit Community Cloud than Spark's partitioned output folder, and it avoids the extra Hadoop `winutils`
binaries Spark needs to *write* files on Windows. All processing above was done in Spark.

| Output | Content | Used by |
|--------|---------|---------|
| `movies_processed.parquet` | one clean row per movie, features + target | 02 EDA, 03 ML, dashboard |
| `genre_stats.parquet` | Spark aggregation per genre | 02 EDA, dashboard |
| `yearly_stats.parquet` | Spark aggregation per release year | 02 EDA, dashboard |
| `movie_ratings_summary.parquet` | joined MovieLens rating aggregates (post-release) | 02 EDA |

In [23]:
import json
outputs = {
    "movies_processed.parquet": final_df,
    "genre_stats.parquet": genre_stats,
    "yearly_stats.parquet": yearly_stats,
    "movie_ratings_summary.parquet": movie_ratings,
}
for name, sdf in outputs.items():
    pdf = sdf.toPandas()
    pdf.to_parquet(PROCESSED_DIR / name, index=False)
    print(f"{name:32s} {len(pdf):6d} rows  {(PROCESSED_DIR / name).stat().st_size/1e6:5.2f} MB")

# Small metadata file so later stages know which genre columns exist
meta = {"genres": GENRES, "genre_columns": GENRE_COLS, "runtime_median": runtime_median,
        "row_count": n_rows, "cleaning_steps": step_counts}
(PROCESSED_DIR / "processing_metadata.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(step_counts, indent=2))

d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


movies_processed.parquet          44861 rows  12.86 MB


d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


genre_stats.parquet                  20 rows   0.01 MB


d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


yearly_stats.parquet                133 rows   0.01 MB


d:\Big Data\Movie-Popularity-Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


movie_ratings_summary.parquet      9004 rows   0.39 MB
{
  "raw": 45466,
  "valid_id": 45463,
  "deduplicated": 45433,
  "released_with_date": 44899,
  "valid_popularity": 44861
}


### Processed dataset schema (handoff to the next stages)

| Column(s) | Meaning | Role |
|-----------|---------|------|
| `id`, `imdb_id`, `title`, `overview` | identifiers & text | display only |
| `release_date`, `release_year`, `release_month`, `release_decade` | release timing | features (year, month) |
| `original_language` | ISO-639-1 code | feature (categorical) |
| `genres_str`, `primary_genre`, `num_genres`, `genre_*` | genres (pipe-joined, first genre, count, flags) | features (`num_genres`, `genre_*`) |
| `budget`, `budget_known`, `log_budget` | budget in USD (null = unknown) | features (`budget_known`, `log_budget`) |
| `runtime`, `runtime_imputed` | minutes (median-imputed) | feature (`runtime`) |
| `has_collection`, `num_companies`, `num_countries`, `is_us_production` | production info | features |
| `revenue`, `vote_count`, `vote_average` | post-release outcomes | **EDA only** |
| `popularity`, `log_popularity` | TMDB popularity score (snapshot ~2017) | **target** |

In [24]:
spark.stop()
print("Spark session stopped. Next: notebooks/02_eda.ipynb")

Spark session stopped. Next: notebooks/02_eda.ipynb
